In [1]:
import random
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    classification_report,
    f1_score,
    average_precision_score
)

/Users/canhu/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
train_df = pd.read_csv('../data/model_training_data.csv')
train_df.head()

,title,selftext,Assignee,feas_pa,feas_ka,feas_cr,feas_sr
0,"Accepted a Job, Relocated, and Then Got My Off...","In other words, they filled the role internall...",Overlap,1.0,1.0,3.0,1.0
1,I Am a Fraud and ChatGPT Is My Brain,literally everything. uni? fake. career pat...,Overlap,4.0,4.0,4.0,1.0
2,Wow interviews suck more now,"This makes you look bad"" and I replied ""if you...",Overlap,1.0,1.0,1.0,2.0
3,Think I'm about to turn Netflix down. Am I crazy?,Did all 7 interviews down and got an offer. 5...,Overlap,1.0,1.0,1.0,1.0
4,"The bar is absolutely, insanely high.",Interviewed at a unicorn tech company for inte...,Overlap,1.0,1.0,1.0,1.0


In [3]:
# Convert to binary labels (0 for <=3, 1 for >3)
train_df['feas_pa'] = (train_df['feas_pa'] > 3).astype(int)
train_df['feas_ka'] = (train_df['feas_ka'] > 3).astype(int)
train_df['feas_cr'] = (train_df['feas_cr'] > 3).astype(int)
train_df['feas_sr'] = (train_df['feas_sr'] > 3).astype(int)
train_df.head()

,title,selftext,Assignee,feas_pa,feas_ka,feas_cr,feas_sr
0,"Accepted a Job, Relocated, and Then Got My Off...","In other words, they filled the role internall...",Overlap,0,0,0,0
1,I Am a Fraud and ChatGPT Is My Brain,literally everything. uni? fake. career pat...,Overlap,1,1,1,0
2,Wow interviews suck more now,"This makes you look bad"" and I replied ""if you...",Overlap,0,0,0,0
3,Think I'm about to turn Netflix down. Am I crazy?,Did all 7 interviews down and got an offer. 5...,Overlap,0,0,0,0
4,"The bar is absolutely, insanely high.",Interviewed at a unicorn tech company for inte...,Overlap,0,0,0,0


In [4]:
# Print class distribution percentages
print("Class distribution for feas_pa:")
print(train_df['feas_pa'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_ka:")
print(train_df['feas_ka'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_cr:")
print(train_df['feas_cr'].value_counts(normalize=True) * 100)
print("\nClass distribution for feas_sr:")
print(train_df['feas_sr'].value_counts(normalize=True) * 100)

Class distribution for feas_pa:
feas_pa
0    80.13245
1    19.86755
Name: proportion, dtype: float64

Class distribution for feas_ka:
feas_ka
0    94.039735
1     5.960265
Name: proportion, dtype: float64

Class distribution for feas_cr:
feas_cr
0    88.741722
1    11.258278
Name: proportion, dtype: float64

Class distribution for feas_sr:
feas_sr
0    99.006623
1     0.993377
Name: proportion, dtype: float64


In [5]:
test_df = pd.read_csv('../data/form_responses.csv')
test_df.head()

,timestamp,consent,demo_gender,demo_area_of_study,demo_honours_class,demo_job_search_status,demo_num_prior_interns,demo_financial_urgency,feas_pa1,feas_pa2,...,feas_sr3,feas_sr4,bhv_apps_4_wks,bhv_itvs_4_wks,bhv_job_update_check_daily,ac2,bhv_app_duration,bhv_app_status_check_daily,bhv_app_avoidance_weekly,bhv_job_search_exp
0,2/24/2026 8:47:57,I agree,Male,Computing and Data,Honours (Highest Distinction),Actively applying,1-2,5,5,2,...,1,2,31+,2 - 3,1 - 2,0,<10 min,1 - 2,1 - 2,Not knowing what the companies want. Sometimes...
1,2/24/2026 11:51:05,I agree,Female,Computing and Data,Honours (Distinction),Actively applying,1-2,3,5,4,...,4,2,6 - 15,1,1 - 2,0,10 - 30 min,0,3 - 5,I dl being on LinkedIn sometimes it’s a luck g...
2,2/24/2026 11:55:42,I agree,Female,Design & Engineering,Honours (Highest Distinction),Actively applying,1-2,2,4,2,...,4,2,16 - 30,0,3 - 5,0,10 - 30 min,1 - 2,1 - 2,1. Not making good progress - not trying hard ...
3,2/24/2026 12:07:31,I agree,Male,Design & Engineering,Honours (Distinction),Actively applying,1-2,5,2,2,...,4,4,0,1,3 - 5,0,10 - 30 min,1 - 2,0,1. Social circle: I feel worried when I hear o...
4,2/24/2026 12:08:09,I agree,Female,Computing and Data,Honours (Distinction),Secured but still actively applying,3 and above,2,4,4,...,2,4,16 - 30,1,1 - 2,0,10 - 30 min,1 - 2,3 - 5,Seeing people get to big companies feels a lit...


In [6]:
# Compute average score for 4 columns: feas_pa1, feas_pa2, feas_pa3, feas_pa4
test_df['feas_pa'] = test_df[['feas_pa1', 'feas_pa2', 'feas_pa3', 'feas_pa4', 'feas_pa5']].mean(axis=1)
test_df['feas_ka'] = test_df[['feas_ka1', 'feas_ka2', 'feas_ka3', 'feas_ka4']].mean(axis=1)
test_df['feas_cr'] = test_df[['feas_cr1', 'feas_cr2', 'feas_cr3', 'feas_cr4']].mean(axis=1)
test_df['feas_sr'] = test_df[['feas_sr1', 'feas_sr2', 'feas_sr3', 'feas_sr4']].mean(axis=1)

In [7]:
# Convert to binary labels (0 for <=3, 1 for >3)
test_df['feas_pa'] = (test_df['feas_pa'] > 3).astype(int)
test_df['feas_ka'] = (test_df['feas_ka'] > 3).astype(int)
test_df['feas_cr'] = (test_df['feas_cr'] > 3).astype(int)
test_df['feas_sr'] = (test_df['feas_sr'] > 3).astype(int)
test_df.head()

,timestamp,consent,demo_gender,demo_area_of_study,demo_honours_class,demo_job_search_status,demo_num_prior_interns,demo_financial_urgency,feas_pa1,feas_pa2,...,bhv_job_update_check_daily,ac2,bhv_app_duration,bhv_app_status_check_daily,bhv_app_avoidance_weekly,bhv_job_search_exp,feas_pa,feas_ka,feas_cr,feas_sr
0,2/24/2026 8:47:57,I agree,Male,Computing and Data,Honours (Highest Distinction),Actively applying,1-2,5,5,2,...,1 - 2,0,<10 min,1 - 2,1 - 2,Not knowing what the companies want. Sometimes...,0,0,0,0
1,2/24/2026 11:51:05,I agree,Female,Computing and Data,Honours (Distinction),Actively applying,1-2,3,5,4,...,1 - 2,0,10 - 30 min,0,3 - 5,I dl being on LinkedIn sometimes it’s a luck g...,1,1,0,0
2,2/24/2026 11:55:42,I agree,Female,Design & Engineering,Honours (Highest Distinction),Actively applying,1-2,2,4,2,...,3 - 5,0,10 - 30 min,1 - 2,1 - 2,1. Not making good progress - not trying hard ...,0,1,1,1
3,2/24/2026 12:07:31,I agree,Male,Design & Engineering,Honours (Distinction),Actively applying,1-2,5,2,2,...,3 - 5,0,10 - 30 min,1 - 2,0,1. Social circle: I feel worried when I hear o...,0,1,0,1
4,2/24/2026 12:08:09,I agree,Female,Computing and Data,Honours (Distinction),Secured but still actively applying,3 and above,2,4,4,...,1 - 2,0,10 - 30 min,1 - 2,3 - 5,Seeing people get to big companies feels a lit...,1,1,1,1


## 1. TF-IDF + Logistic Regression

In [11]:
TARGET_COLS = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']

def prepare_text(df):
    return (df['title'].fillna('') + ' ' + df['selftext'].fillna('')).values

X_train = prepare_text(train_df)
y_train = train_df[TARGET_COLS].values

X_test = test_df['bhv_job_search_exp']
y_test = test_df[TARGET_COLS].values

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print("Shapes of datasets:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Shapes of datasets:
X_train: (241,)
y_train: (241, 4)
X_val: (61,)
y_val: (61, 4)
X_test: (55,)
y_test: (55, 4)


In [12]:
# TF-IDF is inside pipeline → fitted per CV fold
# Logistic Regression wrapped with OneVsRest

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', OneVsRestClassifier(
        LogisticRegression(
            solver='liblinear',
            class_weight='balanced',
            max_iter=1000
        )
    ))
])

In [16]:
# Important: we cannot directly do multi-label stratified CV easily with sklearn
# Practical workaround: 1) Tune using one representative label (e.g. feas_pa); 2) Then retrain full model on all labels

param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1,1), (1,2)],
    'clf__estimator__C': [0.1, 1, 5]
}

# Use most balanced label ('feas_pa') for CV
y_strat = y_train[:, 0]  

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1',
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_strat)  # only for tuning

print("Best params:", grid.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 1); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.0s
[CV] END clf__estimator__C=0.1, tfidf__max_features=5000, tfidf__ngram_range=(1, 2); total time=   0.1s
[CV] END clf__estimator__C=0.1, tfidf__max_features=10000, tfidf__ngram_range=(1, 1); total time=   0.0s
[C

In [17]:
# Train final model on all labels with best params
best_pipeline = grid.best_estimator_

best_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, stop_words='english')),
                ('clf',
                 OneVsRestClassifier(estimator=LogisticRegression(C=0.1,
                                                                  class_weight='balanced',
                                                                  max_iter=1000,
                                                                  solver='liblinear')))])

In [18]:
# Prediction
y_prob_val = best_pipeline.predict_proba(X_val)
y_prob_test = best_pipeline.predict_proba(X_test)

In [23]:
# Threshold tuning (on validation set to prevent data leakage)
def tune_threshold(y_true, y_prob):
    if np.sum(y_true) == 0:
        return 0.5  # default threshold

    best_t, best_f1 = 0.5, 0
    for t in np.linspace(0.05, 0.5, 20):
        preds = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


thresholds = []

for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], y_prob_val[:, i])
    thresholds.append(t)

print("Tuned thresholds:", thresholds)

y_pred_tuned = (y_prob_test >= thresholds).astype(int)

Tuned thresholds: [0.5, 0.4763157894736842, 0.5, 0.5]


In [27]:
# Evaluation (per label)
for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (TFIDF + LR)=====")
    print(classification_report(y_test[:, i], y_pred_tuned[:, i], digits=4, zero_division=0))


===== feas_pa (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.4615    0.9231    0.6154        26
           1     0.3333    0.0345    0.0625        29

    accuracy                         0.4545        55
   macro avg     0.3974    0.4788    0.3389        55
weighted avg     0.3939    0.4545    0.3239        55


===== feas_ka (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.3778    0.9444    0.5397        18
           1     0.9000    0.2432    0.3830        37

    accuracy                         0.4727        55
   macro avg     0.6389    0.5938    0.4613        55
weighted avg     0.7291    0.4727    0.4343        55


===== feas_cr (TFIDF + LR)=====
              precision    recall  f1-score   support

           0     0.4182    1.0000    0.5897        23
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.4182        55
   macro avg     0.2091    0.

In [28]:
# Evaluation (overall)
macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
micro_f1 = f1_score(y_test, y_pred_tuned, average='micro')

print("Macro F1:", macro_f1)
print("Micro F1:", micro_f1)

Macro F1: 0.11136968085106383
Micro F1: 0.145985401459854


## 2. Sentence Embeddings + LightGBM

In [47]:
TARGET_COLS = ['feas_pa', 'feas_ka', 'feas_cr', 'feas_sr']

def prepare_text(df):
    return (df['title'].fillna('') + ' ' + df['selftext'].fillna('')).values

X_train = prepare_text(train_df)
y_train = train_df[TARGET_COLS].values

X_test = test_df['bhv_job_search_exp']
y_test = test_df[TARGET_COLS].values

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print("Shapes of datasets:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Shapes of datasets:
X_train: (241,)
y_train: (241, 4)
X_val: (61,)
y_val: (61, 4)
X_test: (55,)
y_test: (55, 4)


In [30]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

X_train_emb = embedder.encode(X_train, batch_size=16, show_progress_bar=True)
X_val_emb = embedder.encode(X_val, batch_size=16, show_progress_bar=True)
X_test_emb = embedder.encode(X_test, batch_size=16, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2005.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 4/4 [00:00<00:00, 12.66it/s]


In [36]:
# Compute imbalance weight
def compute_scale_pos_weight(y):
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    return neg / (pos + 1e-6)

# Random parameter sampler
param_dist = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 400, 800],
    "min_child_samples": [5, 10, 20],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0]
}

def sample_params(param_dist, n_iter=20):
    keys = list(param_dist.keys())
    sampled = []
    
    for _ in range(n_iter):
        params = {k: random.choice(param_dist[k]) for k in keys}
        sampled.append(params)
    
    return sampled


import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning
)

In [48]:
best_params_per_label = []

for label_idx, col in enumerate(TARGET_COLS):
    print(f"\n==============================")
    print(f"Tuning for {col}")
    print(f"==============================")

    y_label = y_train[:, label_idx]

    # To handle extreme imbalance, we check the minimum class count and adjust n_splits accordingly
    min_class_count = np.bincount(y_label).min()

    if min_class_count < 2:
        print(f"Skipping {col} (too few samples)")
        continue
    n_splits = min(5, min_class_count)

    # Stratify per label
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    scale_pos_weight = compute_scale_pos_weight(y_label)
    param_list = sample_params(param_dist, n_iter=20)

    best_score = -1
    best_params = None

    for params in param_list:
        fold_scores = []

        for train_idx, val_idx in skf.split(X_train_emb, y_label):
            X_tr, X_val = X_train_emb[train_idx], X_train_emb[val_idx]
            y_tr, y_va = y_label[train_idx], y_label[val_idx]

            model = lgb.LGBMClassifier(
                **params,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )

            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_va)],
                eval_metric="average_precision",
                callbacks=[lgb.early_stopping(30, verbose=False)]
            )

            y_prob = model.predict_proba(X_val)[:, 1]
            score = average_precision_score(y_va, y_prob)
            fold_scores.append(score)

        mean_score = np.mean(fold_scores)

        print(f"PR-AUC: {mean_score:.4f} | Params: {params}")

        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"\nBest PR-AUC: {best_score:.4f}")
    print(f"Best Params: {best_params}")

    best_params_per_label.append(best_params)


Tuning for feas_pa
PR-AUC: 0.4393 | Params: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 400, 'min_child_samples': 20, 'subsample': 0.7, 'colsample_bytree': 0.7}
PR-AUC: 0.4620 | Params: {'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 400, 'min_child_samples': 5, 'subsample': 0.9, 'colsample_bytree': 0.7}
PR-AUC: 0.4782 | Params: {'num_leaves': 15, 'learning_rate': 0.1, 'n_estimators': 800, 'min_child_samples': 10, 'subsample': 0.9, 'colsample_bytree': 0.7}
PR-AUC: 0.4669 | Params: {'num_leaves': 15, 'learning_rate': 0.1, 'n_estimators': 200, 'min_child_samples': 20, 'subsample': 0.9, 'colsample_bytree': 0.9}
PR-AUC: 0.4524 | Params: {'num_leaves': 15, 'learning_rate': 0.1, 'n_estimators': 200, 'min_child_samples': 10, 'subsample': 1.0, 'colsample_bytree': 1.0}
PR-AUC: 0.3963 | Params: {'num_leaves': 31, 'learning_rate': 0.1, 'n_estimators': 400, 'min_child_samples': 5, 'subsample': 1.0, 'colsample_bytree': 0.9}
PR-AUC: 0.4460 | Params: {'num_leaves': 15, 'learni

In [49]:
# Train final model
models = []

for label_idx, col in enumerate(TARGET_COLS):
    print(f"\nTraining final model for {col}")

    y_label = y_train[:, label_idx]
    scale_pos_weight = compute_scale_pos_weight(y_label)

    model = lgb.LGBMClassifier(
        **best_params_per_label[label_idx],
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    model.fit(X_train_emb, y_label)
    models.append(model)


Training final model for feas_pa

Training final model for feas_ka

Training final model for feas_cr

Training final model for feas_sr


In [50]:
# Prediction
num_models = len(models)

y_prob_val = np.zeros((len(y_val), num_models))
y_prob_test = np.zeros((len(y_test), num_models))

for i, model in enumerate(models):
    y_prob_val[:, i] = model.predict_proba(X_val_emb)[:, 1]
    y_prob_test[:, i] = model.predict_proba(X_test_emb)[:, 1]

In [ ]:
# Threshold tuning on validation set to prevent data leakage
thresholds = []

for i in range(len(TARGET_COLS)):
    t = tune_threshold(y_val[:, i], y_prob_val[:, i])
    thresholds.append(t)

print("Best thresholds:", thresholds)

# y_pred = np.zeros_like(y_prob, dtype=int)
for i in range(len(TARGET_COLS)):
    y_pred_tuned[:, i] = (y_prob_test[:, i] >= thresholds[i]).astype(int)

Best thresholds: [0.05, 0.5, 0.05, 0.5]


In [53]:
# Evaluation (per label)
for i, col in enumerate(TARGET_COLS):
    print(f"\n===== {col} (Embedding + LightGBM) =====")
    print(classification_report(y_test[:, i], y_pred_tuned[:, i], digits=4, zero_division=0))


===== feas_pa (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.5000    0.8077    0.6176        26
           1     0.6154    0.2759    0.3810        29

    accuracy                         0.5273        55
   macro avg     0.5577    0.5418    0.4993        55
weighted avg     0.5608    0.5273    0.4928        55


===== feas_ka (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.3333    1.0000    0.5000        18
           1     1.0000    0.0270    0.0526        37

    accuracy                         0.3455        55
   macro avg     0.6667    0.5135    0.2763        55
weighted avg     0.7818    0.3455    0.1990        55


===== feas_cr (Embedding + LightGBM) =====
              precision    recall  f1-score   support

           0     0.4524    0.8261    0.5846        23
           1     0.6923    0.2812    0.4000        32

    accuracy                         0.5091       

In [54]:
# Evaluation (overall)
macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
micro_f1 = f1_score(y_test, y_pred_tuned, average='micro')

print("Macro F1:", macro_f1)
print("Micro F1:", micro_f1)

Macro F1: 0.20839598997493736
Micro F1: 0.23841059602649006
